	 # _______________________ START: VISUALISATION WFLOW OUTPUT  _______________________ 



In [37]:
using CSV, Dates, Tables, CairoMakie, GLMakie,  NCDatasets, Dates


In [46]:
Path_Home = @__DIR__
cd(Path_Home)

include("Julia//Parameters.jl")

@assert isfile(joinpath(Path_Wflow_NetCDF_Output))
WflowOutput = NCDatasets.NCDataset(Path_Wflow_NetCDF_Output)

ΔX = 5 # m
ΔY = 5 # m

StartYear    = 2021
StartMonth    = 01
StartDay      = 01

StartDate = Date(StartYear, StartMonth, StartDay)


2021-01-01

In [69]:
Precipitation        = Array(WflowOutput["Precipitation"])
Stemflow             = Array(WflowOutput["Stemflow"])
PotentialEvaporation = Array(WflowOutput["PotentialEvaporation"])
Throughfall          = Array(WflowOutput["Throughfall"])
Interception         = Array(WflowOutput["Interception"])
CanopyStorage        = Array(WflowOutput["CanopyStorage"])

Nx, Ny, Nt = size(Precipitation)

Precipitation_Average        = fill(0.0, Nt)
Stemflow_Average             = fill(0.0, Nt)
PotentialEvaporation_Average = fill(0.0, Nt)
Throughfall_Average          = fill(0.0, Nt)
Interception_Average         = fill(0.0, Nt)
CanopyStorage_Average        = fill(0.0, Nt)
ReachSoil_Average       = fill(0.0, Nt)
NotMissing                   = fill(false, Nx, Ny)
DatesWflow                   = fill(Date(StartYear, StartMonth, StartDay), Nt)


# Counting active cells
	CountCell = 0
	 for iX=1:Nx, iY=1:Ny
		if Precipitation[iX, iY, 2] !== missing
			CountCell += 1
			NotMissing[iX, iY] = true
		end
	end

	CatchmentArea = CountCell * ΔX * ΔY  # m

	printstyled("	Number of grids = $CountCell , CatchmentSize = $CatchmentArea  m² \n", color = :green)

	DatesWflow[1] = StartDate

for iT=1:Nt
	 for iX=1:Nx
		for iY=1:Ny
			if NotMissing[iX, iY]
				Precipitation_Average[iT] += Precipitation[iX, iY, iT]
				Stemflow_Average[iT] += Stemflow[iX, iY, iT]
				PotentialEvaporation_Average[iT] += PotentialEvaporation[iX, iY, iT]
				Throughfall_Average[iT] += Throughfall[iX, iY, iT]
				Interception_Average[iT] += Interception[iX, iY, iT]
				CanopyStorage_Average[iT] += CanopyStorage[iX, iY, iT]
			end
		end
	end # for iX=1:Nx, iY=1:Ny

   Precipitation_Average[iT]        = Precipitation_Average[iT] / CountCell
   Stemflow_Average[iT]             = Stemflow_Average[iT] / CountCell
   PotentialEvaporation_Average[iT] = PotentialEvaporation_Average[iT] / CountCell
   Throughfall_Average[iT]          = Throughfall_Average[iT] / CountCell
   Interception_Average[iT]         = Interception_Average[iT] / CountCell
   CanopyStorage_Average[iT]        = CanopyStorage_Average[iT] / CountCell

	ReachSoil_Average[iT] = Precipitation_Average[iT] - Interception_Average[iT]

	DatesWflow[iT] +=  Dates.Day(iT)
end # for iT=1:Nt




	Number of grids = 324770 , CatchmentSize = 8119250  m² 


In [71]:
# Water balance
# [Precipitation_Average] - Interception_Average[iT] = Stemflow_Average[iT] + Throughfall_Average[iT]

Path_AverageFluxes_Csv₀ = joinpath(Path_Root, Path_WflowModelOutput)
mkpath(Path_AverageFluxes_Csv₀)

Path_AverageFluxes_Csv = joinpath(Path_AverageFluxes_Csv₀, Filename_WflowCatchementAverage_Csv)

Header = ["Date", "Precipitation[mm]", "PotentialEvaporation[mm]", "Stemflow[mm]", "Throughfall[mm]",  "Interception[mm]", "ReachSoil[mm]"]
CSV.write(Path_AverageFluxes_Csv, Tables.table([DatesWflow Precipitation_Average PotentialEvaporation_Average Stemflow_Average Throughfall_Average Interception_Average ReachSoil_Average]), writeheader=true, header=Header, bom=true)

InterceptionLost = 100.0 * sum(Interception_Average) / sum(Precipitation_Average)

printstyled("	Interception lost = $(floor(InterceptionLost)) %  \n", color = :green)


	Interception lost = 11.0 %  
